# Стадия 5. Подбор гиперпараметров: сетка против байесовской оптимизации

**Данные:** `data/processed/train_model.parquet` (результат стадии 1); описание набора — `README.md`.
**Предыдущие этапы:** `Stage_3_classification.ipynb` — базовая модель и порог решения;
`Stage_4_polynomial_regularization.ipynb` — полиномиальные признаки и три вида штрафа.

**Цель стадии.** Сравнить два способа подбора гиперпараметров на одной и той же задаче:
полный перебор сетки (`GridSearchCV` из scikit-learn) и байесовскую оптимизацию (Optuna).
Вопрос не в том, какой инструмент лучше вообще, а в том, что каждый даёт на нашей задаче
и при каком размере пространства поиска разница становится заметной.

**Зачем это нужно.** В стадии 4 перебор шёл по одному параметру `C` — восемь значений,
сорок обучений модели. На таком пространстве сетка оптимальна: она быстрее и показывает
всю картину целиком. Но как только параметров становится три (степень полинома, вид штрафа,
сила штрафа), полная сетка разрастается до сотни комбинаций, и часть из них заведомо
бесполезна. Здесь и появляется смысл в инструменте, который выбирает следующую точку
по результатам уже проверенных.

**Целевая переменная — бинарная `Event`:** 0 — пациент жив со своей печенью, 1 — трансплантация
или смерть. Критерий подбора — лог-потеря (качество вероятностей). Итоговое качество на
контрольной части оценивается набором метрик: ROC-AUC, PR-AUC, метрика Брайера, лог-потеря,
F-мера, полнота, точность, доля верных ответов.


## **1. Импорты и настройки**

In [1]:
import os
import time

import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             f1_score, log_loss, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

pd.set_option('display.width', 200)
# Единое зерно случайности для всего ноутбука: выборки, разбиения, кросс-валидация.
# Значение не меняется между прогонами и стадиями, поэтому результаты сравнимы.
RANDOM_STATE = 42

# Optuna по умолчанию печатает строку на каждую попытку — для отчёта это лишний шум
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Версия Optuna:', optuna.__version__)


Версия Optuna: 5.1.0


## **2. Данные и подготовка матриц**

Разбиение и масштабирование повторяют стадии 3 и 4 без изменений: та же доля контрольной
части, та же стратификация по классу события, то же зерно случайности. Иначе результаты
нельзя сравнивать с предыдущими стадиями.


In [2]:
# Корень проекта: папка с data/, найденная вверх по дереву от текущей директории
def find_project_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, 'data')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError('Папка data/ не найдена вверх по дереву от ' + d)
        d = parent


PROJECT_ROOT = find_project_root()
df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_model.parquet'))
assert df.shape == (7905, 16), 'Ожидается матрица 7 905 x 16 из стадии 1'
assert df.isna().sum().sum() == 0, 'В матрице модели не должно быть пропусков'

X = df.drop(columns=['Event']).astype(float)
y = df['Event'].astype(int)
assert X.shape[1] == 15, 'В матрице должно быть 15 отобранных признаков'
assert sorted(y.unique()) == [0, 1], 'Цель должна быть бинарной'

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)   # fit только на train
X_te = scaler.transform(X_test)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Размеры выборок:', X_tr.shape[0], 'train /', X_te.shape[0], 'test')
print('Событий в контрольной части:', int(y_test.sum()), 'из', len(y_test))


def evaluate(model, X_eval, y_eval, tag):
    # Набор метрик бинарной классификации — тот же, что в стадиях 3 и 4.
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]
    return {
        'вариант': tag,
        'ROC-AUC': roc_auc_score(y_eval, y_proba),
        'PR-AUC': average_precision_score(y_eval, y_proba),
        'метрика Брайера': brier_score_loss(y_eval, y_proba),
        'лог-потеря': log_loss(y_eval, y_proba),
        'F-мера': f1_score(y_eval, y_pred),
        'полнота': recall_score(y_eval, y_pred),
        'точность': precision_score(y_eval, y_pred),
        'доля верных ответов': accuracy_score(y_eval, y_pred),
    }


Размеры выборок: 5928 train / 1977 test
Событий в контрольной части: 735 из 1977


### Полиномиальные признаки второй степени

Степень 1 — исходные 15 признаков. Степень 2 — те же признаки с квадратами и произведениями
(132 столбца после удаления квадратов бинарных признаков, как в стадии 4). Обе матрицы
готовятся один раз: внутри перебора остаётся только обучение модели.

In [3]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_poly = poly.fit_transform(X_tr)
X_te_poly = poly.transform(X_te)
poly_names = list(poly.get_feature_names_out(X.columns))

# Квадраты бинарных признаков линейно зависимы от самих признаков — удаляем (стадия 4)
BINARY_FEATURES = ['Sex_M', 'Hepatomegaly_Y', 'Spiders_Y']
square_names = {f'{name}^2' for name in BINARY_FEATURES}
keep_idx = [i for i, name in enumerate(poly_names) if name not in square_names]
X_tr_poly = X_tr_poly[:, keep_idx]
X_te_poly = X_te_poly[:, keep_idx]

# Второе масштабирование: штраф должен ложиться на столбцы равномерно
poly_scaler = StandardScaler()
X_tr_p = poly_scaler.fit_transform(X_tr_poly)
X_te_p = poly_scaler.transform(X_te_poly)

assert X_tr_p.shape == (5928, 132), 'Ожидается 132 столбца: 135 минус 3 квадрата бинарных'
assert X_te_p.shape == (1977, 132)
print(f'Степень 1: {X_tr.shape[1]} столбцов')
print(f'Степень 2: {X_tr_p.shape[1]} столбцов')


def features_for(degree):
    # Возвращает готовые матрицы для выбранной степени полинома.
    return (X_tr, X_te) if degree == 1 else (X_tr_p, X_te_p)

Степень 1: 15 столбцов
Степень 2: 132 столбцов


## **3. Пространство поиска**

Перебираем три параметра одновременно:

| Параметр | Что задаёт | Значения |
|---|---|---|
| `degree` | степень полинома: 1 — прямые связи, 2 — квадраты и произведения | 1, 2 |
| `l1_ratio` | вид штрафа: 0 — L2, 1 — L1, между — ElasticNet | 0, 0.25, 0.5, 0.75, 1 |
| `C` | сила штрафа (обратная: больше `C` — слабее штраф) | 0.003 … 10 |

Один параметр `l1_ratio` заменяет три отдельных вида штрафа: это современный способ
задавать регуляризацию в scikit-learn, где `penalty` объявлен устаревшим. Смысл прежний:
`l1_ratio = 0` — L2, `l1_ratio = 1` — L1, промежуточные значения — ElasticNet.

In [4]:
DEGREES = [1, 2]
L1_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]
C_MIN, C_MAX = 1e-3, 10.0
N_TRIALS = 25

n_grid = len(DEGREES) * len(L1_GRID) * len(C_GRID)
n_folds = CV.get_n_splits()

print(f'Полная сетка: {len(DEGREES)} x {len(L1_GRID)} x {len(C_GRID)} = {n_grid} комбинаций')
print(f'Обучений модели при полном переборе: {n_grid} x {n_folds} = {n_grid * n_folds}')
print(f'Байесовская оптимизация: {N_TRIALS} попыток x {n_folds} = {N_TRIALS * n_folds} обучений')
print(f'Экономия по числу обучений: в {n_grid / N_TRIALS:.1f} раза')

Полная сетка: 2 x 5 x 8 = 80 комбинаций
Обучений модели при полном переборе: 80 x 5 = 400
Байесовская оптимизация: 25 попыток x 5 = 125 обучений
Экономия по числу обучений: в 3.2 раза


## **4. Нативный инструмент: полный перебор сетки**

`GridSearchCV` проверяет каждую комбинацию на каждой из пяти частей обучающей выборки.
Плюс метода — гарантия, что лучшее значение в заданных границах не пропущено. Минус —
цена растёт линейно: добавили один параметр с пятью значениями, получили в пять раз
больше обучений.

Степень полинома перебирается внешним циклом: для каждой степени своя матрица признаков.

In [5]:
def make_model(l1_ratio, C):
    # saga поддерживает L1 и ElasticNet; lbfgs — только L2.
    return LogisticRegression(solver='saga', l1_ratio=l1_ratio, C=C,
                              max_iter=5000, tol=1e-3, random_state=RANDOM_STATE)


grid_rows = []
grid_best = None

started = time.time()
for degree in DEGREES:
    M_tr, _ = features_for(degree)
    search = GridSearchCV(
        make_model(0.0, 1.0),
        {'l1_ratio': L1_GRID, 'C': C_GRID},
        scoring='neg_log_loss', cv=CV, n_jobs=-1,
    )
    search.fit(M_tr, y_train)
    score = -search.best_score_
    grid_rows.append({'степень': degree, **search.best_params_, 'лог-потеря (CV)': score})
    if grid_best is None or score < grid_best['лог-потеря (CV)']:
        grid_best = {'degree': degree, **search.best_params_, 'лог-потеря (CV)': score}

grid_elapsed = time.time() - started
grid_trainings = n_grid * n_folds

display(pd.DataFrame(grid_rows).set_index('степень').round(4))
print(f'Лучшая комбинация сетки: степень {grid_best["degree"]}, '
      f'l1_ratio = {grid_best["l1_ratio"]}, C = {grid_best["C"]}')
print(f'Лог-потеря на CV: {grid_best["лог-потеря (CV)"]:.4f}')
print(f'Время перебора: {grid_elapsed:.1f} с, обучений модели: {grid_trainings}')

,C,l1_ratio,лог-потеря (CV)
степень,,,
1,0.10,0.0,0.4204
2,0.03,0.5,0.4101


Лучшая комбинация сетки: степень 2, l1_ratio = 0.5, C = 0.03
Лог-потеря на CV: 0.4101
Время перебора: 14.3 с, обучений модели: 400


## **5. Байесовская оптимизация: Optuna**

Optuna не перебирает сетку, а строит модель зависимости «параметры → качество» и по ней
выбирает следующую точку. Первые попытки случайны, дальше алгоритм (TPE — оценка плотности
по хорошим и плохим результатам) смещает поиск туда, где результат был лучше.

Три особенности, которые важны для отчёта:

- **`C` ищется на непрерывной шкале** (логарифмической), а не по фиксированным точкам.
  Поэтому Optuna может остановиться на значении, которого в сетке нет (например, `C = 0.079`).
- **`seed = RANDOM_STATE` (42) обязателен.** Без него прогон невоспроизводим: повторный запуск даст другие
  параметры. С фиксированным зерном результат повторяется.
- **`n_trials` фиксирован.** Число обучений известно заранее, поэтому сравнение с сеткой
  по цене честное.

In [6]:
def objective(trial):
    # Одна попытка = одна комбинация гиперпараметров, оценённая по кросс-валидации.
    degree = trial.suggest_categorical('degree', DEGREES)
    l1_ratio = trial.suggest_categorical('l1_ratio', L1_GRID)
    C = trial.suggest_float('C', C_MIN, C_MAX, log=True)

    M_tr, _ = features_for(degree)
    model = make_model(l1_ratio, C)
    scores = cross_val_score(model, M_tr, y_train, cv=CV,
                             scoring='neg_log_loss', n_jobs=-1)
    return -scores.mean()


study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))

started = time.time()
study.optimize(objective, n_trials=N_TRIALS)
optuna_elapsed = time.time() - started

optuna_best = study.best_params
optuna_score = study.best_value
optuna_trainings = N_TRIALS * n_folds

print(f'Лучшая комбинация Optuna: степень {optuna_best["degree"]}, '
      f'l1_ratio = {optuna_best["l1_ratio"]}, C = {optuna_best["C"]:.4f}')
print(f'Лог-потеря на CV: {optuna_score:.4f}')
print(f'Время поиска: {optuna_elapsed:.1f} с, обучений модели: {optuna_trainings}')

Лучшая комбинация Optuna: степень 2, l1_ratio = 0.5, C = 0.0292
Лог-потеря на CV: 0.4101
Время поиска: 9.0 с, обучений модели: 125


## **6. Сравнение двух инструментов**

In [7]:
comparison = pd.DataFrame([
    {
        'инструмент': 'полный перебор сетки (GridSearchCV)',
        'степень': grid_best['degree'],
        'l1_ratio': grid_best['l1_ratio'],
        'C': round(grid_best['C'], 4),
        'лог-потеря (CV)': round(grid_best['лог-потеря (CV)'], 4),
        'обучений модели': grid_trainings,
        'время, с': round(grid_elapsed, 1),
    },
    {
        'инструмент': 'байесовская оптимизация (Optuna)',
        'степень': optuna_best['degree'],
        'l1_ratio': optuna_best['l1_ratio'],
        'C': round(optuna_best['C'], 4),
        'лог-потеря (CV)': round(optuna_score, 4),
        'обучений модели': optuna_trainings,
        'время, с': round(optuna_elapsed, 1),
    },
])

display(comparison.set_index('инструмент'))

delta = optuna_score - grid_best['лог-потеря (CV)']
print(f'Разница по лог-потере (Optuna минус сетка): {delta:+.4f}')
print(f'Экономия обучений у Optuna: в {grid_trainings / optuna_trainings:.1f} раза')
print(f'Экономия времени: в {grid_elapsed / optuna_elapsed:.1f} раза')

,степень,l1_ratio,C,лог-потеря (CV),обучений модели,"время, с"
инструмент,,,,,,
полный перебор сетки (GridSearchCV),2,0.5,0.0300,0.4101,400,14.3
байесовская оптимизация (Optuna),2,0.5,0.0292,0.4101,125,9.0


Разница по лог-потере (Optuna минус сетка): +0.0000
Экономия обучений у Optuna: в 3.2 раза
Экономия времени: в 1.6 раза


### Вывод по сравнению инструментов

Сравнение идёт по двум осям: качеству и цене.

**По качеству инструменты совпали полностью.** Оба нашли вторую степень полинома и
`l1_ratio = 0.5` — ту же комбинацию, что стадия 4 выбрала для ElasticNet. Лог-потеря на
кросс-валидации 0.4101 у обоих, разница меньше 0.00001 — это ноль в пределах точности
вычисления. Различилась только сила штрафа: сетка остановилась на узле 0.03, Optuna
уточнила значение внутри того же интервала — 0.0292. На таком пространстве выигрыша
по качеству ждать неоткуда: сетка содержит точку, близкую к оптимуму, и находит её.

**По цене разница в 3.2 раза:** 400 обучений модели у сетки против 125 у Optuna. Время в
этом прогоне — 14.3 против 9.0 секунд (в 1.6 раза): выигрыш меньше, потому что у Optuna
есть накладные расходы на построение своей модели зависимости. Точное время зависит от
загрузки машины, поэтому основной мерой экономии выбрано число обучений.

**Практический вывод для проекта:** пока параметров один-два, `GridSearchCV` остаётся
правильным выбором — он проще, быстрее и показывает всю картину (в стадии 4 по таблице
видно, что оптимум `C` внутренний, а не на краю диапазона). Optuna выигрывает, когда
пространство вырастает до трёх и более параметров, как здесь, или когда одно обучение
занимает минуты: там экономия в разы по числу обучений превращается в экономию часов.


## **7. Проверка лучшей модели на контрольной части**

Подбор шёл только по обучающей части. Контрольная часть до этого момента не участвовала
и используется один раз — для итоговой оценки обеих моделей.

In [8]:
test_rows = []

for tag, params in [('лучшая по сетке', grid_best),
                    ('лучшая по Optuna', {'degree': optuna_best['degree'],
                                          'l1_ratio': optuna_best['l1_ratio'],
                                          'C': optuna_best['C']})]:
    M_tr, M_te = features_for(params['degree'])
    model = make_model(params['l1_ratio'], params['C'])
    model.fit(M_tr, y_train)
    test_rows.append(evaluate(model, M_te, y_test, tag))

# Точка отсчёта — базовая модель стадии 3 без полиномов
base_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
base_model.fit(X_tr, y_train)
test_rows.insert(0, evaluate(base_model, X_te, y_test, 'базовая (стадия 3)'))

test_table = pd.DataFrame(test_rows).set_index('вариант')
display(test_table.round(4))

# Для сверки: лучшие варианты стадии 4 на контрольной части
STAGE4_REF = {'полином 2 ст. + L1 (C = 0.1)': (0.8819, 0.4162),
              'полином 2 ст. + ElasticNet (C = 0.03)': (0.8834, 0.4133)}
for name, (auc, ll) in STAGE4_REF.items():
    print(f'{name}: ROC-AUC {auc:.4f}, лог-потеря {ll:.4f}')


,ROC-AUC,PR-AUC,метрика Брайера,лог-потеря,F-мера,полнота,точность,доля верных ответов
вариант,,,,,,,,
базовая (стадия 3),0.8788,0.8132,0.1333,0.4257,0.7307,0.6830,0.7856,0.8128
лучшая по сетке,0.8834,0.8249,0.1308,0.4133,0.7404,0.7102,0.7733,0.8149
лучшая по Optuna,0.8835,0.8251,0.1308,0.4132,0.7404,0.7102,0.7733,0.8149


полином 2 ст. + L1 (C = 0.1): ROC-AUC 0.8819, лог-потеря 0.4162
полином 2 ст. + ElasticNet (C = 0.03): ROC-AUC 0.8834, лог-потеря 0.4133


### Вывод по контрольной части

Обе модели лучше базовой по всем метрикам вероятностного прогноза:

| Вариант | ROC-AUC | PR-AUC | Метрика Брайера | Лог-потеря | F-мера | Доля верных ответов |
|---|---|---|---|---|---|---|
| базовая (стадия 3) | 0.8788 | 0.8132 | 0.1333 | 0.4257 | 0.7307 | 0.8128 |
| лучшая по сетке | 0.8834 | 0.8249 | 0.1308 | 0.4133 | 0.7404 | 0.8149 |
| лучшая по Optuna | **0.8835** | **0.8251** | 0.1308 | **0.4132** | 0.7404 | 0.8149 |

Расхождение между инструментами — в четвёртом знаке: 0.0001 по ROC-AUC и PR-AUC, 0.0001
по лог-потере. Такая разница меньше шага между фолдами кросс-валидации, поэтому выбрать
инструмент по контрольной части нельзя — и не нужно.

Существенно другое: подбор по трём параметрам **не превзошёл** результат стадии 4.
Обе модели воспроизвели вариант «полином 2 ст. + ElasticNet» с теми же 0.8834 и 0.4133.
Optuna уточнила силу штрафа (0.0292 вместо 0.03), но это не дало ничего: 0.8835 против
0.8834 — округление, а не улучшение. Значит, добавление степени полинома в пространство
поиска не открыло новых возможностей: данные уже отработаны на стадии 4. Ценность стадии 5
методическая — она показывает, как сравнивать инструменты подбора и когда какой уместен.


## **8. Выводы**

### Что сделано

Сравнены два инструмента подбора гиперпараметров на одной задаче, с одинаковой
кросс-валидацией, одинаковой метрикой (лог-потеря) и одинаковым разбиением данных.
Цель — бинарная `Event` (0 — жив, 1 — трансплантация или смерть).

| Вопрос | Ответ |
|---|---|
| Даёт ли Optuna лучшее качество, чем сетка? | Нет. Оба пришли в одну точку: степень 2, `l1_ratio = 0.5`, лог-потеря на CV 0.4101 |
| Даёт ли Optuna экономию? | Да, в 3.2 раза меньше обучений модели (125 против 400) при том же результате |
| Когда нужна сетка? | Один-два параметра: проще, быстрее, видна вся картина |
| Когда нужна Optuna? | Три и более параметра, либо дорогое обучение, либо непрерывные шкалы |

### Что это меняет для проекта

Ничего в итоговой модели: лучший вариант остался на стадии 4, подбор по трём параметрам
его не превзошёл — расхождения в четвёртом знаке. Стадия 5 добавляет к отчёту обоснованный
ответ на вопрос «каким инструментом подбирать параметры, если пространство вырастет» —
например, при переходе к нелинейным моделям с пятью-шестью параметрами.

### Глоссарий терминов

| Термин | Что означает | Зачем нужен |
|---|---|---|
| **Гиперпараметр** | Настройка метода, которую задаёт человек до обучения: сила штрафа, степень полинома | От неё зависит качество; подбирается по кросс-валидации, а не по контрольной части |
| **Полный перебор сетки** | Проверка всех комбинаций из заранее заданного списка значений | Гарантирует, что лучшее значение в границах не пропущено |
| **Байесовская оптимизация** | Выбор следующей точки по результатам уже проверенных | Экономит обучения, когда комбинаций много |
| **`TPESampler`** | Алгоритм Optuna: строит оценку плотности хороших и плохих результатов | Смещает поиск к перспективной области вместо случайного тыка |
| **`n_trials`** | Число попыток в Optuna | Задаёт цену поиска: одна попытка — одна кросс-валидация |
| **`seed`** | Зерно случайности | Без него прогон невоспроизводим: повтор даст другие параметры |
| **`l1_ratio`** | Доля L1 в штрафе: 0 — L2, 1 — L1, между — ElasticNet | Один параметр вместо трёх видов штрафа |
| **`C`** | Сила штрафа, обратная `alpha`: больше `C` — слабее штраф | Основной параметр регуляризации логистической регрессии |
| **Кросс-валидация** | Обучение и проверка на нескольких разбиениях обучающей части | Даёт устойчивую оценку без использования контрольной выборки |
| **Лог-потеря** | Штраф за уверенность в неверном ответе | Критерий подбора: оценивает качество вероятностей, а не только классов |
| **ROC-AUC** | Вероятность, что случайное событие получило оценку выше случайного «нуля» | Показывает разделяющую способность модели без привязки к порогу |
| **PR-AUC** | Площадь под кривой «полнота — точность» | Чувствительна к качеству на редком классе события |
| **Метрика Брайера** | Средний квадрат отклонения предсказанной вероятности от факта | Оценивает калибровку вероятностей |
